In [2]:
from peft import LoraConfig, get_peft_model, TaskType
# 1교시 라이브러리 추가
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


using devcies : cpu


In [3]:
# 1교시에 사용한 데이터 train, test
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

from glob import glob
import pandas as pd
import re
df = pd.read_csv(glob(path+'/*')[0])
df['review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣a-zA-Z0-9\s]', '', x.strip().lower()))

df['sentiment'].apply(lambda x : x.strip().lower())
t = [{'text':review, 'label':sentiment} for review, sentiment in zip(df['review'].to_list(),df['sentiment'].to_list())]
train_data = t[:500]
test_data = t[-100:]
print(len(train_data), len(test_data))

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
500 100


In [4]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

# LoRA 설정
peft_config= LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r = 8,          # 저차원 랭크의 크기
    lora_alpha=32,  # 가중치 스케일 상수
    lora_dropout=0.1,
    target_modules=['q','v']    
)

# PEFT 모델로 변환
model = get_peft_model(model, peft_config=peft_config)
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5335.84it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


(77305216, 344064, 0.445072166928555)

In [5]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_fuction(example):
    # T5모델에 맞게 토큰화
    inputs = [ f"Review: {text}\nSentiment: Answer with either positive or negative."  for text in example['text'] ]
    model_input = tokenizer(inputs,max_length=128, truncation=True)
    # 라벨을 토큰화
    lables = tokenizer(text_target=example['label'],max_length=128, truncation=True)
    model_input['labels'] = lables['input_ids']
    return model_input
# 2. DataSet객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑()
tokenized_train = train_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)

Map: 100%|██████████| 100/100 [00:00<00:00, 4564.19 examples/s]


In [6]:
# training Arguments 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_lora_results',
    eval_strategy='epoch',
    learning_rate=1e-4,
    num_train_epochs=2,
    predict_with_generate=True,
    logging_steps=2
)
# Data Collator, Trainer 객체
data_collator = DataCollatorForSeq2Seq(tokenizer,model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,train_dataset=tokenized_train, eval_dataset=tokenized_test,
    processing_class=tokenizer, data_collator = data_collator
)
# 학습시간 측정
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'Fine-turning completed : {training_time:.2f} seconds') 

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.003809,2.280673
2,1.738707,1.342414


c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-turning completed : 154.86 seconds
